<a href="https://colab.research.google.com/github/emmanuelmassawe/breast-cancer-predictions-with-pytorch/blob/main/breast_cancer_predictions_with_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# importing the necessary libraries

In [40]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset,DataLoader
import pandas as pd

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


loaing the dataset

In [16]:
data = load_breast_cancer()
X, y = data.data, data.target

In [45]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


we train_test and split the dataset

In [17]:
X_train,X_test ,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42 , stratify=y)


we check the shape of the data

In [18]:
print(X_train.shape)
print(X_test.shape)

(455, 30)
(114, 30)


we perform feature scaling


In [19]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

we change to tensor arrays

In [20]:
X_train = torch.tensor(X_train,dtype=torch.float32).to(device)
X_test = torch.tensor(X_test,dtype=torch.float32).to(device)

y_train = torch.tensor(y_train, dtype = torch.long).to(device)
y_test = torch.tensor(y_test, dtype = torch.long).to(device)



we create a dataset and a data loader

In [21]:
train_dataset = TensorDataset(X_train,y_train)
test_dataset = TensorDataset(X_test,y_test)

train_loader = DataLoader(train_dataset, batch_size= 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size= 32, shuffle = False)

creating the Neural Network Architecture


In [26]:
class Nn(nn.Module):
  def __init__(self):
    super().__init__()

    self.network = nn.Sequential(
        nn.Linear(30,64),
        nn.ReLU(),
        nn.Linear(64,32),
        nn.ReLU(),
        nn.Linear(32,2)
    )
  def forward(self,x):
    return self.network(x)

model = Nn().to(device)
print(model)

Nn(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=2, bias=True)
  )
)


Apply the loss function

In [31]:
loss_fn = nn.CrossEntropyLoss().to(device)

we apply an optimizer

In [32]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.001
)

train the model

In [34]:
epoch = 20

for i in range(epoch):
  model.train()
  for X_batch,y_batch in train_loader:
    optimizer.zero_grad()
    y_pred = model(X_batch)
    loss = loss_fn(y_pred,y_batch)


    loss.backward()
    optimizer.step()

    if (i+1) % 10 == 0:
      print(f"Epoch {i+1}/{epoch}, Loss: {loss.item():.4f}")



Epoch 10/20, Loss: 0.0000
Epoch 10/20, Loss: 0.0010
Epoch 10/20, Loss: 0.0002
Epoch 10/20, Loss: 0.0005
Epoch 10/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0003
Epoch 10/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0000
Epoch 10/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0000
Epoch 10/20, Loss: 0.0187
Epoch 10/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0004
Epoch 10/20, Loss: 0.0012
Epoch 20/20, Loss: 0.0001
Epoch 20/20, Loss: 0.0001
Epoch 20/20, Loss: 0.0003
Epoch 20/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0003
Epoch 20/20, Loss: 0.0001
Epoch 20/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0008
Epoch 20/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0001
Epoch 20/20, Loss: 0.0187
Epoch 20/20, Loss: 0.0001
Epoch 20/20, Loss: 0.0001
Epoch 20/20, Loss: 0.0000


model evaluation


In [37]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
  for X_batch,y_batch in test_loader:
    y_pred = model(X_batch)
    pred = torch.argmax(y_pred , dim = 1)

    correct += (pred == y_batch).sum().item()
    total += y_batch.size(0)

accuracy = correct / total
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 95.61%


saving the model

In [47]:
torch.save(model.state_dict(),'model.pth')